In [1]:
import pandas as pd
import numpy as np
import os

data_dir = '../datasets'

In [2]:
# Load all relevant source files
customers_df = pd.read_csv(f'{data_dir}/olist_customers_dataset.csv')
orders_df = pd.read_csv(f'{data_dir}/olist_orders_dataset.csv')
order_items_df = pd.read_csv(f'{data_dir}/olist_order_items_dataset.csv')
order_payments_df = pd.read_csv(f'{data_dir}/olist_order_payments_dataset.csv')
order_reviews_df = pd.read_csv(f'{data_dir}/olist_order_reviews_dataset.csv')
products_df = pd.read_csv(f'{data_dir}/olist_products_dataset.csv')
product_translation_df = pd.read_csv(f'{data_dir}/product_category_name_translation.csv')

In [3]:
orders_df.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='object')

In [4]:
# Orders timestamps
date_cols = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 
             'order_delivered_customer_date', 'order_estimated_delivery_date']
for col in date_cols:
    orders_df[col] = pd.to_datetime(orders_df[col])

# Review timestamps
order_reviews_df['review_creation_date'] = pd.to_datetime(order_reviews_df['review_creation_date'])
order_reviews_df['review_answer_timestamp'] = pd.to_datetime(order_reviews_df['review_answer_timestamp'])

In [5]:
## find the cutoff date
## if we consider a 90 day period rather than the 30 days
## then date will be Jul 1 

In [6]:
# How many orders fall in Jul 1 - Sep 29, 2018?
target_mask = (orders_df['order_purchase_timestamp'] >= '2018-07-01') & (orders_df['order_purchase_timestamp'] < '2018-10-01')
print(orders_df[target_mask].shape[0])

12820


In [7]:
feature_mask = orders_df['order_purchase_timestamp'] < '2018-07-01'
target_mask = (orders_df['order_purchase_timestamp'] >= '2018-07-01') & (orders_df['order_purchase_timestamp'] < '2018-10-01')

feature_customers = orders_df[feature_mask].merge(customers_df, on='customer_id')['customer_unique_id'].unique()
target_customers = orders_df[target_mask].merge(customers_df, on='customer_id')['customer_unique_id'].unique()

overlap = len(set(target_customers) & set(feature_customers))
print(f"Feature window customers: {len(feature_customers)}")
print(f"Target window customers: {len(target_customers)}")
print(f"Repeat customers (in both): {overlap}")
print(f"Positive rate: {overlap/len(feature_customers)*100:.2f}%")

Feature window customers: 83748
Target window customers: 12645
Repeat customers (in both): 298
Positive rate: 0.36%


In [8]:
CUTOFF_DATE = pd.Timestamp('2018-07-01')
TARGET_END = pd.Timestamp('2018-10-01')

### Gropup : 1 - Order count & status features

In [9]:
orders_customers = orders_df.merge(customers_df[['customer_id', 'customer_unique_id']], on='customer_id', how='left')

feature_orders = orders_customers[orders_customers['order_purchase_timestamp'] < CUTOFF_DATE].copy()

#Total order count per customer
order_counts = feature_orders.groupby('customer_unique_id')['order_id'].count().reset_index()
order_counts.columns = ['customer_unique_id', 'num_orders']

# Order count by status
# groupby TWO columns: customer AND status, count orders in each combination
# then unstack turns each status value into its own column
status_counts = feature_orders.groupby(['customer_unique_id', 'order_status'])['order_id'].count().unstack(fill_value=0)
status_counts.columns = ['num_orders_' + col for col in status_counts.columns]
status_counts = status_counts.reset_index()

group1_features = order_counts.merge(status_counts, on='customer_unique_id', how='left')

print(f"Shape: {group1_features.shape}")
print(f"Columns: {list(group1_features.columns)}")
print(group1_features.head())

Shape: (83748, 10)
Columns: ['customer_unique_id', 'num_orders', 'num_orders_approved', 'num_orders_canceled', 'num_orders_created', 'num_orders_delivered', 'num_orders_invoiced', 'num_orders_processing', 'num_orders_shipped', 'num_orders_unavailable']
                 customer_unique_id  num_orders  num_orders_approved  \
0  0000366f3b9a7992bf8c76cfdf3221e2           1                    0   
1  0000b849f77a49e4a4ce2b2a4ca5be3f           1                    0   
2  0000f46a3911fa3c0805444483337064           1                    0   
3  0000f6ccb0745a6a4b88665a16c9f078           1                    0   
4  0004aac84e0df4da2b147fca70cf8255           1                    0   

   num_orders_canceled  num_orders_created  num_orders_delivered  \
0                    0                   0                     1   
1                    0                   0                     1   
2                    0                   0                     1   
3                    0                   0

In [10]:
print(group1_features[['num_orders', 'num_orders_approved', 'num_orders_canceled', 'num_orders_created', 
                         'num_orders_delivered', 'num_orders_invoiced', 'num_orders_processing', 
                         'num_orders_shipped', 'num_orders_unavailable']].sum())

num_orders                86617
num_orders_approved           2
num_orders_canceled         481
num_orders_created            5
num_orders_delivered      83968
num_orders_invoiced         278
num_orders_processing       300
num_orders_shipped          999
num_orders_unavailable      584
dtype: int64


In [11]:
# users who have non-zero values in interesting columns
print(group1_features[group1_features['num_orders_canceled'] > 0].head())
print(f"\nUsers with cancellations: {(group1_features['num_orders_canceled'] > 0).sum()}")
print(f"Users with multiple orders: {(group1_features['num_orders'] > 1).sum()}")

                   customer_unique_id  num_orders  num_orders_approved  \
116  0058f300f57d7b93c477a131a59b36c3           2                    0   
203  009b0127b727ab0ba422f6d9604487c7           1                    0   
319  00f0b70fdcb8a6e1671b52a2472bd41f           1                    0   
416  013f66477aa3210eb05fec3fa184de33           1                    0   
627  01ea7dfdac01a4e8fbe2902b73510b20           2                    0   

     num_orders_canceled  num_orders_created  num_orders_delivered  \
116                    1                   0                     1   
203                    1                   0                     0   
319                    1                   0                     0   
416                    1                   0                     0   
627                    1                   0                     1   

     num_orders_invoiced  num_orders_processing  num_orders_shipped  \
116                    0                      0                

### Group 2: Money features

In [12]:
feature_orders.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'customer_unique_id'],
      dtype='object')

In [13]:
payment_features = feature_orders.merge(order_payments_df, on='order_id',how='left')

payment_agg = payment_features.groupby('customer_unique_id').agg(
    
    total_payment_value = ('payment_value', 'sum'),
    total_payment_sequential = ('payment_sequential', 'sum'),
    avg_payment_installments = ('payment_installments', 'mean'),

).reset_index()

payment_type_split = payment_features.groupby(['customer_unique_id', 'payment_type'])['payment_value'].sum().unstack(fill_value=0)
payment_type_split.columns = ['tot_pymt_' + col for col in payment_type_split.columns]
payment_type_split = payment_type_split.reset_index()

group2_features = payment_agg.merge(payment_type_split, on='customer_unique_id', how='left')


print(f"Shape: {group2_features.shape}")
print(f"Columns: {list(group2_features.columns)}")
print(group2_features.head())

Shape: (83748, 8)
Columns: ['customer_unique_id', 'total_payment_value', 'total_payment_sequential', 'avg_payment_installments', 'tot_pymt_boleto', 'tot_pymt_credit_card', 'tot_pymt_debit_card', 'tot_pymt_voucher']
                 customer_unique_id  total_payment_value  \
0  0000366f3b9a7992bf8c76cfdf3221e2               141.90   
1  0000b849f77a49e4a4ce2b2a4ca5be3f                27.19   
2  0000f46a3911fa3c0805444483337064                86.22   
3  0000f6ccb0745a6a4b88665a16c9f078                43.62   
4  0004aac84e0df4da2b147fca70cf8255               196.89   

   total_payment_sequential  avg_payment_installments  tot_pymt_boleto  \
0                       1.0                       8.0              0.0   
1                       1.0                       1.0              0.0   
2                       1.0                       8.0              0.0   
3                       1.0                       4.0              0.0   
4                       1.0                       6.0 

### Group 3: Item & product features

In [14]:
item_features = feature_orders.merge(order_items_df, on='order_id', how='left')

order_size = item_features.groupby('order_id')['order_item_id'].count().reset_index()
order_size.columns = ['order_id', 'order_size']
item_features = item_features.merge(order_size, on='order_id', how='left')

group3_features = item_features.groupby('customer_unique_id').agg(
    num_products=('order_item_id', 'count'),
    num_sellers=('seller_id', 'nunique'),
    avg_order_size=('order_size', 'mean'),
    tot_order_price=('price', 'sum'),
    avg_order_price=('price', 'mean'),
    tot_order_freight_value=('freight_value', 'sum'),
    avg_order_freight_value=('freight_value', 'mean'),
).reset_index()

print(f"Shape: {group3_features.shape}")
print(f"Columns: {list(group3_features.columns)}")
print(group3_features.head())

Shape: (83748, 8)
Columns: ['customer_unique_id', 'num_products', 'num_sellers', 'avg_order_size', 'tot_order_price', 'avg_order_price', 'tot_order_freight_value', 'avg_order_freight_value']
                 customer_unique_id  num_products  num_sellers  \
0  0000366f3b9a7992bf8c76cfdf3221e2             1            1   
1  0000b849f77a49e4a4ce2b2a4ca5be3f             1            1   
2  0000f46a3911fa3c0805444483337064             1            1   
3  0000f6ccb0745a6a4b88665a16c9f078             1            1   
4  0004aac84e0df4da2b147fca70cf8255             1            1   

   avg_order_size  tot_order_price  avg_order_price  tot_order_freight_value  \
0             1.0           129.90           129.90                    12.00   
1             1.0            18.90            18.90                     8.29   
2             1.0            69.00            69.00                    17.22   
3             1.0            25.99            25.99                    17.63   
4           

### Group 4: Review features

In [15]:
order_reviews_df.columns

Index(['review_id', 'order_id', 'review_score', 'review_comment_title',
       'review_comment_message', 'review_creation_date',
       'review_answer_timestamp'],
      dtype='object')

In [16]:
reviews_features = feature_orders.merge(order_reviews_df, on='order_id', how='left')

reviews_features['review_comment_title'] = reviews_features['review_comment_title'].fillna('')
reviews_features['review_comment_message'] = reviews_features['review_comment_message'].fillna('')

reviews_features['title_length'] = reviews_features['review_comment_title'].str.len()
reviews_features['body_length'] = reviews_features['review_comment_message'].str.len()

group4_features = reviews_features.groupby('customer_unique_id').agg(
    
    num_reviews=('review_id', 'count'),
    avg_review_score=('review_score', 'mean'),
    avg_review_title_length=('title_length', 'mean'),
    avg_review_body_length=('body_length', 'mean')
    
).reset_index()

group4_features.fillna(0, inplace=True)

print(f"Shape: {group4_features.shape}")
print(f"Columns: {list(group4_features.columns)}")
print(group4_features.head())

Shape: (83748, 5)
Columns: ['customer_unique_id', 'num_reviews', 'avg_review_score', 'avg_review_title_length', 'avg_review_body_length']
                 customer_unique_id  num_reviews  avg_review_score  \
0  0000366f3b9a7992bf8c76cfdf3221e2            1               5.0   
1  0000b849f77a49e4a4ce2b2a4ca5be3f            1               4.0   
2  0000f46a3911fa3c0805444483337064            1               3.0   
3  0000f6ccb0745a6a4b88665a16c9f078            1               4.0   
4  0004aac84e0df4da2b147fca70cf8255            1               5.0   

   avg_review_title_length  avg_review_body_length  
0                     15.0                   111.0  
1                      0.0                     0.0  
2                      0.0                     0.0  
3                      0.0                    12.0  
4                      0.0                     0.0  


### Group 5: Recency features

In [17]:
order_recency = feature_orders.groupby('customer_unique_id').agg(
    last_purchase=('order_purchase_timestamp', 'max'),
    last_shipped=('order_delivered_carrier_date', 'max'),
    last_delivered=('order_delivered_customer_date', 'max')
).reset_index()


order_recency['days_since_last_purchase'] = (CUTOFF_DATE - order_recency['last_purchase']).dt.days
order_recency['days_since_last_shipped'] = (CUTOFF_DATE - order_recency['last_shipped']).dt.days
order_recency['days_since_last_delivered'] = (CUTOFF_DATE - order_recency['last_delivered']).dt.days

order_recency = order_recency.drop(columns=['last_purchase', 'last_shipped', 'last_delivered'])

reviews_features['review_creation_date'] = pd.to_datetime(reviews_features['review_creation_date'])

review_recency = reviews_features.groupby('customer_unique_id').agg(
    last_review=('review_creation_date', 'max')
).reset_index()

review_recency['days_since_last_review'] = (CUTOFF_DATE - review_recency['last_review']).dt.days
review_recency = review_recency.drop(columns=['last_review'])

group5_features = order_recency.merge(review_recency, on='customer_unique_id', how='left')

print(f"Shape: {group5_features.shape}")
print(f"Columns: {list(group5_features.columns)}")
print(group5_features.head())

Shape: (83748, 5)
Columns: ['customer_unique_id', 'days_since_last_purchase', 'days_since_last_shipped', 'days_since_last_delivered', 'days_since_last_review']
                 customer_unique_id  days_since_last_purchase  \
0  0000366f3b9a7992bf8c76cfdf3221e2                        51   
1  0000b849f77a49e4a4ce2b2a4ca5be3f                        54   
2  0000f46a3911fa3c0805444483337064                       477   
3  0000f6ccb0745a6a4b88665a16c9f078                       261   
4  0004aac84e0df4da2b147fca70cf8255                       228   

   days_since_last_shipped  days_since_last_delivered  days_since_last_review  
0                     49.0                       45.0                    45.0  
1                     52.0                       51.0                    51.0  
2                    474.0                      451.0                   451.0  
3                    260.0                      241.0                   241.0  
4                    226.0                      2

### Group 6: Preference Features

In [18]:
def get_favorite(series):
    # .mode() finds the most frequent value(s)
    modes = series.mode()
    if len(modes) > 0:
        return modes.iloc[0]
    return "Unknown"

items_with_products = order_items_df.merge(products_df, on='product_id', how='left')
items_with_english = items_with_products.merge(product_translation_df, on='product_category_name', how='left')

user_items = feature_orders[['order_id', 'customer_unique_id']].merge(items_with_english, on='order_id', how='inner')

pref_category_df = user_items.groupby('customer_unique_id').agg(
    pref_product_category=('product_category_name_english', get_favorite)
).reset_index() 

user_payments = feature_orders[['order_id', 'customer_unique_id']].merge(order_payments_df, on='order_id', how='inner')
pref_payment_df = user_payments.groupby('customer_unique_id').agg(
    pref_payment_type=('payment_type', get_favorite)
).reset_index()

group6_features = pref_category_df.merge(pref_payment_df, on='customer_unique_id', how='outer')

group6_features.fillna('Unknown', inplace=True)

print(f"Shape: {group6_features.shape}")
print(f"Columns: {list(group6_features.columns)}")
print(group6_features.head())

Shape: (83748, 3)
Columns: ['customer_unique_id', 'pref_product_category', 'pref_payment_type']
                 customer_unique_id pref_product_category pref_payment_type
0  0000366f3b9a7992bf8c76cfdf3221e2        bed_bath_table       credit_card
1  0000b849f77a49e4a4ce2b2a4ca5be3f         health_beauty       credit_card
2  0000f46a3911fa3c0805444483337064            stationery       credit_card
3  0000f6ccb0745a6a4b88665a16c9f078             telephony       credit_card
4  0004aac84e0df4da2b147fca70cf8255             telephony       credit_card


## Merging All Features

In [19]:
user_features = group1_features \
    .merge(group2_features, on='customer_unique_id', how='left') \
    .merge(group3_features, on='customer_unique_id', how='left') \
    .merge(group4_features, on='customer_unique_id', how='left') \
    .merge(group5_features, on='customer_unique_id', how='left') \
    .merge(group6_features, on='customer_unique_id', how='left')

print(f"Shape: {user_features.shape}")
print(f"Columns: {list(user_features.columns)}")
print(user_features.isnull().sum()[user_features.isnull().sum() > 0])

Shape: (83748, 34)
Columns: ['customer_unique_id', 'num_orders', 'num_orders_approved', 'num_orders_canceled', 'num_orders_created', 'num_orders_delivered', 'num_orders_invoiced', 'num_orders_processing', 'num_orders_shipped', 'num_orders_unavailable', 'total_payment_value', 'total_payment_sequential', 'avg_payment_installments', 'tot_pymt_boleto', 'tot_pymt_credit_card', 'tot_pymt_debit_card', 'tot_pymt_voucher', 'num_products', 'num_sellers', 'avg_order_size', 'tot_order_price', 'avg_order_price', 'tot_order_freight_value', 'avg_order_freight_value', 'num_reviews', 'avg_review_score', 'avg_review_title_length', 'avg_review_body_length', 'days_since_last_purchase', 'days_since_last_shipped', 'days_since_last_delivered', 'days_since_last_review', 'pref_product_category', 'pref_payment_type']
avg_payment_installments        1
tot_pymt_boleto                 1
tot_pymt_credit_card            1
tot_pymt_debit_card             1
tot_pymt_voucher                1
avg_order_price            

### Null Analysis & Imputation Strategy

Before imputing, let's understand WHY each null exists:

| Feature Group | Null Count | Root Cause | Strategy | Rationale |
|---|---|---|---|---|
| Payment columns (installments, boleto, credit_card, etc.) | 1 each | Single order with no payment record in source | Fill with 0 | No payment record = no payment made through that channel |
| avg_order_price, avg_order_freight | 620 | Orders with no items in order_items table (verified in exploration notebook) | Fill with median | Real customers with missing item-level data; median is robust to right-skewed distributions |
| days_since_last_shipped | 1,473 | Orders never shipped (canceled/created/processing status) | Fill with column max | Never shipped = longest possible time since shipment; preserves signal that these users had unfulfilled orders |
| days_since_last_delivered | 2,482 | Orders never delivered (same as above, larger count because delivery happens after shipping) | Fill with column max | Same logic as shipped |
| days_since_last_review | 651 | Users who never submitted a review | Fill with column max | No review = least engaged; max recency captures this |

In [20]:
# Payment nulls: fill with 0
payment_cols = ['avg_payment_installments', 'tot_pymt_boleto', 'tot_pymt_credit_card', 'tot_pymt_debit_card', 'tot_pymt_voucher']
user_features[payment_cols] = user_features[payment_cols].fillna(0)

In [21]:
# Item nulls: fill with median
item_cols = ['avg_order_price', 'avg_order_freight_value']
for col in item_cols:
    user_features[col] = user_features[col].fillna(user_features[col].median())


In [22]:
# Recency nulls: fill with max value (never happened = longest time ago)
recency_cols = ['days_since_last_shipped', 'days_since_last_delivered', 'days_since_last_review']
for col in recency_cols:
    user_features[col] = user_features[col].fillna(user_features[col].max())

In [23]:
print("Null check after imputation:")
print(user_features.isnull().sum().sum())
print(f"\nDataset shape: {user_features.shape}")
print(f"Features: {user_features.shape[1] - 1} + 1 ID column")

Null check after imputation:
0

Dataset shape: (83748, 34)
Features: 33 + 1 ID column


## Creating Target Variables

In [24]:
target_orders = orders_customers[
    (orders_customers['order_purchase_timestamp'] >= CUTOFF_DATE) & 
    (orders_customers['order_purchase_timestamp'] < TARGET_END)
].copy()

# Exclude canceled and unavailable orders
target_orders = target_orders[~target_orders['order_status'].isin(['canceled', 'unavailable'])]

target_items = target_orders.merge(order_items_df, on='order_id', how='inner')

order_level_value = target_items.groupby(['order_id', 'customer_unique_id']).agg(
    order_total_price=('price', 'sum')
).reset_index()

# Average Order Value per CUSTOMER
customer_targets = order_level_value.groupby('customer_unique_id').agg(
    y_order_value=('order_total_price', 'mean')
).reset_index()

customer_targets['y_propensity'] = 1

modelling_dataset = user_features.merge(customer_targets, on='customer_unique_id', how='left')

modelling_dataset['y_propensity'] = modelling_dataset['y_propensity'].fillna(0).astype(int)
modelling_dataset['y_order_value'] = modelling_dataset['y_order_value'].fillna(0)

# Apply the Log Transformation to handle skewness
# np.log1p safely calculates log(1 + x)
modelling_dataset['y_order_value_log'] = np.log1p(modelling_dataset['y_order_value'])

print(f"Final Dataset Shape: {modelling_dataset.shape}")
print("Target Class Distribution (y_propensity):")
print(modelling_dataset['y_propensity'].value_counts(normalize=True) * 100)

print(modelling_dataset[['customer_unique_id', 'y_propensity', 'y_order_value', 'y_order_value_log']].head())

Final Dataset Shape: (83748, 37)
Target Class Distribution (y_propensity):
y_propensity
0    99.650141
1     0.349859
Name: proportion, dtype: float64
                 customer_unique_id  y_propensity  y_order_value  \
0  0000366f3b9a7992bf8c76cfdf3221e2             0            0.0   
1  0000b849f77a49e4a4ce2b2a4ca5be3f             0            0.0   
2  0000f46a3911fa3c0805444483337064             0            0.0   
3  0000f6ccb0745a6a4b88665a16c9f078             0            0.0   
4  0004aac84e0df4da2b147fca70cf8255             0            0.0   

   y_order_value_log  
0                0.0  
1                0.0  
2                0.0  
3                0.0  
4                0.0  


## Saving final Dataset

In [25]:
modelling_dataset.to_csv('../processed/modelling_dataset.csv', index=False)
print(f"Saved modelling dataset: {modelling_dataset.shape}")

user_features.to_csv('../processed/user_features.csv', index=False)
print(f"Saved user features: {user_features.shape}")

Saved modelling dataset: (83748, 37)
Saved user features: (83748, 34)


## Feature Engineering Summary

**Feature window:** 2016-09-04 to 2018-07-01 (22 months of history)
**Target window:** 2018-07-01 to 2018-10-01 (90 days)

**Features engineered:** 33 features across 6 groups:
1. **Order counts & status** (10) — total orders + pivoted by status
2. **Monetary / payment splits** (7) — total value, installments, value by payment type
3. **Item & product** (7) — products, sellers, order size, price, freight
4. **Review engagement** (4) — count, score, title length, body length
5. **Recency** (4) — days since last purchase, shipped, delivered, review
6. **Preferences** (2) — preferred product category, preferred payment type

**Target variables:**
- `y_propensity`: binary (0/1) — 0.35% positive rate (293 repeat buyers)
- `y_order_value`: continuous — average order value in target window
- `y_order_value_log`: log(1 + y_order_value) for regression modelling

**Dataset:** 83,748 users × 37 columns (33 features + 1 ID + 3 targets)

In [26]:
## Generate Feature Engineering Documentation
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

wb = Workbook()
ws = wb.active
ws.title = "Feature Documentation"

header_font = Font(name='Arial', bold=True, size=11, color='FFFFFF')
header_fill = PatternFill('solid', fgColor='2F5496')
group_fill = PatternFill('solid', fgColor='D6E4F0')
group_font = Font(name='Arial', bold=True, size=10, color='2F5496')
normal_font = Font(name='Arial', size=10)
thin_border = Border(
    left=Side(style='thin', color='B4C6E7'),
    right=Side(style='thin', color='B4C6E7'),
    top=Side(style='thin', color='B4C6E7'),
    bottom=Side(style='thin', color='B4C6E7')
)

headers = ['Source Table', 'Feature Name', 'Description', 'Type', 'Aggregation Logic', 'Data Journey']
for col, header in enumerate(headers, 1):
    cell = ws.cell(row=1, column=col, value=header)
    cell.font = header_font
    cell.fill = header_fill
    cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    cell.border = thin_border

ws.column_dimensions['A'].width = 22
ws.column_dimensions['B'].width = 32
ws.column_dimensions['C'].width = 55
ws.column_dimensions['D'].width = 14
ws.column_dimensions['E'].width = 45
ws.column_dimensions['F'].width = 38

features = [
    ['', '', 'GROUP 1: ORDER COUNT & STATUS FEATURES', '', '', ''],
    ['Orders + Customers', 'num_orders', 'Total number of orders placed by the customer', 'Numerical', 'count of order_id per customer_unique_id', 'Orders > Customers'],
    ['Orders + Customers', 'num_orders_approved', 'Number of orders with approved status', 'Numerical', 'count where order_status=approved, pivoted via unstack', 'Orders > Customers'],
    ['Orders + Customers', 'num_orders_canceled', 'Number of orders with canceled status', 'Numerical', 'count where order_status=canceled, pivoted via unstack', 'Orders > Customers'],
    ['Orders + Customers', 'num_orders_created', 'Number of orders with created status', 'Numerical', 'count where order_status=created, pivoted via unstack', 'Orders > Customers'],
    ['Orders + Customers', 'num_orders_delivered', 'Number of orders with delivered status', 'Numerical', 'count where order_status=delivered, pivoted via unstack', 'Orders > Customers'],
    ['Orders + Customers', 'num_orders_invoiced', 'Number of orders with invoiced status', 'Numerical', 'count where order_status=invoiced, pivoted via unstack', 'Orders > Customers'],
    ['Orders + Customers', 'num_orders_processing', 'Number of orders with processing status', 'Numerical', 'count where order_status=processing, pivoted via unstack', 'Orders > Customers'],
    ['Orders + Customers', 'num_orders_shipped', 'Number of orders with shipped status', 'Numerical', 'count where order_status=shipped, pivoted via unstack', 'Orders > Customers'],
    ['Orders + Customers', 'num_orders_unavailable', 'Number of orders with unavailable status', 'Numerical', 'count where order_status=unavailable, pivoted via unstack', 'Orders > Customers'],
    ['', '', 'GROUP 2: MONETARY & PAYMENT FEATURES', '', '', ''],
    ['Payments + Orders', 'total_payment_value', 'Total payment value across all customer orders', 'Numerical', 'sum of payment_value per customer_unique_id', 'Payments > Orders > Customers'],
    ['Payments + Orders', 'total_payment_sequential', 'Total count of payment records (multi-payment orders)', 'Numerical', 'sum of payment_sequential per customer_unique_id', 'Payments > Orders > Customers'],
    ['Payments + Orders', 'avg_payment_installments', 'Average installments used across orders', 'Numerical', 'mean of payment_installments per customer_unique_id', 'Payments > Orders > Customers'],
    ['Payments + Orders', 'tot_pymt_boleto', 'Total payment value via boleto (bank slip)', 'Numerical', 'sum of payment_value where type=boleto, pivoted', 'Payments > Orders > Customers'],
    ['Payments + Orders', 'tot_pymt_credit_card', 'Total payment value via credit card', 'Numerical', 'sum of payment_value where type=credit_card, pivoted', 'Payments > Orders > Customers'],
    ['Payments + Orders', 'tot_pymt_debit_card', 'Total payment value via debit card', 'Numerical', 'sum of payment_value where type=debit_card, pivoted', 'Payments > Orders > Customers'],
    ['Payments + Orders', 'tot_pymt_voucher', 'Total payment value via voucher', 'Numerical', 'sum of payment_value where type=voucher, pivoted', 'Payments > Orders > Customers'],
    ['', '', 'GROUP 3: ITEM & PRODUCT FEATURES', '', '', ''],
    ['Order Items + Orders', 'num_products', 'Total number of items purchased', 'Numerical', 'count of order_item_id per customer_unique_id', 'Order Items > Orders > Customers'],
    ['Order Items + Orders', 'num_sellers', 'Number of unique sellers bought from', 'Numerical', 'nunique of seller_id per customer_unique_id', 'Order Items > Orders > Customers'],
    ['Order Items + Orders', 'avg_order_size', 'Average number of items per order', 'Numerical', 'count items per order_id, then mean per customer', 'Order Items > Orders > Customers'],
    ['Order Items + Orders', 'tot_order_price', 'Total item price across all orders', 'Numerical', 'sum of price per customer_unique_id', 'Order Items > Orders > Customers'],
    ['Order Items + Orders', 'avg_order_price', 'Average item price across all orders', 'Numerical', 'mean of price per customer_unique_id', 'Order Items > Orders > Customers'],
    ['Order Items + Orders', 'tot_order_freight_value', 'Total freight cost across all orders', 'Numerical', 'sum of freight_value per customer_unique_id', 'Order Items > Orders > Customers'],
    ['Order Items + Orders', 'avg_order_freight_value', 'Average freight cost per item', 'Numerical', 'mean of freight_value per customer_unique_id', 'Order Items > Orders > Customers'],
    ['', '', 'GROUP 4: REVIEW ENGAGEMENT FEATURES', '', '', ''],
    ['Reviews + Orders', 'num_reviews', 'Total reviews submitted by the customer', 'Numerical', 'count of review_id per customer_unique_id', 'Reviews > Orders > Customers'],
    ['Reviews + Orders', 'avg_review_score', 'Average star rating given (1-5)', 'Numerical', 'mean of review_score per customer_unique_id', 'Reviews > Orders > Customers'],
    ['Reviews + Orders', 'avg_review_title_length', 'Average character length of review titles', 'Numerical', 'str.len() on title (fillna empty), then mean', 'Reviews > Orders > Customers'],
    ['Reviews + Orders', 'avg_review_body_length', 'Average character length of review messages', 'Numerical', 'str.len() on message (fillna empty), then mean', 'Reviews > Orders > Customers'],
    ['', '', 'GROUP 5: RECENCY FEATURES', '', '', ''],
    ['Orders + Customers', 'days_since_last_purchase', 'Days from cutoff to last order purchase', 'Numerical', 'CUTOFF - max(order_purchase_timestamp), .dt.days', 'Orders > Customers'],
    ['Orders + Customers', 'days_since_last_shipped', 'Days from cutoff to last carrier shipment', 'Numerical', 'CUTOFF - max(order_delivered_carrier_date), .dt.days', 'Orders > Customers'],
    ['Orders + Customers', 'days_since_last_delivered', 'Days from cutoff to last customer delivery', 'Numerical', 'CUTOFF - max(order_delivered_customer_date), .dt.days', 'Orders > Customers'],
    ['Reviews + Orders', 'days_since_last_review', 'Days from cutoff to last review submission', 'Numerical', 'CUTOFF - max(review_creation_date), .dt.days', 'Reviews > Orders > Customers'],
    ['', '', 'GROUP 6: PREFERENCE FEATURES', '', '', ''],
    ['Products + Items', 'pref_product_category', 'Most frequently purchased product category', 'Categorical', 'mode of product_category_name_english per customer', 'Products > Items > Orders > Customers'],
    ['Payments + Orders', 'pref_payment_type', 'Most frequently used payment method', 'Categorical', 'mode of payment_type per customer', 'Payments > Orders > Customers'],
]

row = 2
for feature in features:
    is_group = feature[2].startswith('GROUP')
    for col, value in enumerate(feature, 1):
        cell = ws.cell(row=row, column=col, value=value)
        cell.font = group_font if is_group else normal_font
        cell.fill = group_fill if is_group else PatternFill()
        cell.border = thin_border
        cell.alignment = Alignment(vertical='center', wrap_text=True)
    row += 1

# Sheet 2: Feature Selection
ws2 = wb.create_sheet("Feature Selection")
headers2 = ['Dropped Feature', 'Correlated With', 'Correlation (r)', 'Reason for Dropping']
for col, h in enumerate(headers2, 1):
    cell = ws2.cell(row=1, column=col, value=h)
    cell.font = header_font
    cell.fill = header_fill
    cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    cell.border = thin_border

ws2.column_dimensions['A'].width = 28
ws2.column_dimensions['B'].width = 28
ws2.column_dimensions['C'].width = 16
ws2.column_dimensions['D'].width = 60

dropped = [
    ['tot_order_price', 'total_payment_value', 0.986, 'Payment value is actual amount paid; price is a subset'],
    ['days_since_last_review', 'days_since_last_purchase', 0.961, 'Purchase is the primary event; review follows it'],
    ['days_since_last_shipped', 'days_since_last_delivered', 0.949, 'Delivery is the customer-facing milestone'],
    ['avg_order_price', 'total_payment_value', 0.895, 'Redundant once tot_order_price also dropped'],
    ['avg_order_size', 'num_products', 0.913, 'Total product count is more direct'],
    ['avg_order_freight_value', 'tot_order_freight_value', 0.767, 'Total freight is more informative'],
    ['num_orders_delivered', 'num_orders', 0.756, 'Total orders already captures delivery count'],
]

for r, d in enumerate(dropped, 2):
    for col, val in enumerate(d, 1):
        cell = ws2.cell(row=r, column=col, value=val)
        cell.font = normal_font
        cell.border = thin_border
        cell.alignment = Alignment(vertical='center', wrap_text=True)

# Sheet 3: Null Handling
ws3 = wb.create_sheet("Null Handling")
headers3 = ['Feature(s)', 'Null Count', 'Root Cause', 'Strategy', 'Rationale']
for col, h in enumerate(headers3, 1):
    cell = ws3.cell(row=1, column=col, value=h)
    cell.font = header_font
    cell.fill = header_fill
    cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    cell.border = thin_border

ws3.column_dimensions['A'].width = 38
ws3.column_dimensions['B'].width = 12
ws3.column_dimensions['C'].width = 45
ws3.column_dimensions['D'].width = 22
ws3.column_dimensions['E'].width = 60

nulls = [
    ['avg_payment_installments, tot_pymt_*', '1 each', 'Single order with no payment record in source', 'Fill with 0', 'No payment record = no payment through that channel'],
    ['avg_order_price, avg_order_freight_value', '620', 'Orders with no items in order_items table', 'Fill with median', 'Real customers with missing item data; median robust to right-skewed distributions'],
    ['days_since_last_shipped', '1,473', 'Orders never shipped (canceled/created/processing)', 'Fill with column max', 'Never shipped = longest possible time; tree models interpret large value as distant'],
    ['days_since_last_delivered', '2,482', 'Orders never delivered (downstream of shipping)', 'Fill with column max', 'Same logic as shipped; delivery follows shipping so more nulls'],
    ['days_since_last_review', '651', 'Users who never submitted a review', 'Fill with column max', 'No review = least engaged; max recency captures disengagement'],
]

for r, d in enumerate(nulls, 2):
    for col, val in enumerate(d, 1):
        cell = ws3.cell(row=r, column=col, value=val)
        cell.font = normal_font
        cell.border = thin_border
        cell.alignment = Alignment(vertical='center', wrap_text=True)

# Sheet 4: Target Variables
ws4 = wb.create_sheet("Target Variables")
headers4 = ['Target', 'Type', 'Definition', 'Computation Logic', 'Notes']
for col, h in enumerate(headers4, 1):
    cell = ws4.cell(row=1, column=col, value=h)
    cell.font = header_font
    cell.fill = header_fill
    cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    cell.border = thin_border

ws4.column_dimensions['A'].width = 22
ws4.column_dimensions['B'].width = 16
ws4.column_dimensions['C'].width = 45
ws4.column_dimensions['D'].width = 55
ws4.column_dimensions['E'].width = 50

targets = [
    ['y_propensity', 'Binary (0/1)', 'Did customer order in the 90-day target window?', 'Filter orders CUTOFF to TARGET_END, exclude canceled/unavailable, map to customer_unique_id, 1 if present', 'Positive rate: 0.35% (293 / 83,748)'],
    ['y_order_value', 'Continuous ($)', 'Average order value in the target window', 'Sum item prices per order_id, then mean per customer_unique_id; 0 for non-buyers', 'Right-skewed; most buyers $50-$200'],
    ['y_order_value_log', 'Continuous (log)', 'Log-transformed: log(1 + y_order_value)', 'np.log1p(y_order_value); inverse with np.expm1()', 'Handles skew for regression training'],
]

for r, d in enumerate(targets, 2):
    for col, val in enumerate(d, 1):
        cell = ws4.cell(row=r, column=col, value=val)
        cell.font = normal_font
        cell.border = thin_border
        cell.alignment = Alignment(vertical='center', wrap_text=True)

wb.save('../data_reports/Feature_Engineering.xlsx')